# Task 3: Spam Filtering

Условия:
- удалить строки с пропусками;
- split: `test_size=0.3`, `random_state=74`;
- модель: `RandomForestClassifier(n_estimators=5, n_jobs=10, random_state=74)`;
- метрики: macro precision / recall / f1 (округление до 0.001).

In [1]:
import pandas as pd
import scipy.sparse as sp

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 74
TEST_SIZE = 0.3

df = pd.read_csv('data-task-3.csv')
df = df.dropna().reset_index(drop=True)
df.head()

,class,date,from,to,subject,body
0,0,4,info@global-change.com,michelle.lokay@enron.com,next wave energi trade,energi industri profession global chang associ...
1,0,1,info@pmaconference.com,michelle.lokay@enron.com,regist next txu capac auction,regist next txu energi capac auction new regis...
2,0,6,info@pmaconference.com,michelle.lokay@enron.com,merchant power monthli free sampl,merchant power monthli month s issu almost mw ...
3,0,3,bruno@firstconf.com,energynews@fc.ease.lsoft.com,eyeforenergi updat,welcom week s eyeforenergi updat refresh memor...
4,0,1,deanrogers@energyclasses.com,michelle.lokay@enron.com,deriv earli bird til march houston,deriv energi profession two full day april ear...


In [3]:
y = df['class']

def evaluate_predictions(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    return {
        'precision': round(report['macro avg']['precision'], 3),
        'recall': round(report['macro avg']['recall'], 3),
        'f1': round(report['macro avg']['f1-score'], 3),
    }

## 1) TF-IDF по `body`

In [4]:
x_body = df['body'].astype(str)

x_train_body, x_test_body, y_train, y_test = train_test_split(
    x_body,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

vectorizer_1 = TfidfVectorizer()
X_train_1 = vectorizer_1.fit_transform(x_train_body)
X_test_1 = vectorizer_1.transform(x_test_body)

model_1 = RandomForestClassifier(n_estimators=5, n_jobs=10, random_state=RANDOM_STATE)
model_1.fit(X_train_1, y_train)
pred_1 = model_1.predict(X_test_1)

metrics_1 = evaluate_predictions(y_test, pred_1)
metrics_1

{'precision': 0.964, 'recall': 0.963, 'f1': 0.963}

## 2) TF-IDF по `subject + body` + признак дня недели (`date`)

In [5]:
text_2 = (df['subject'].astype(str) + ' ' + df['body'].astype(str)).str.strip()
x_2 = pd.DataFrame({'date': df['date'], 'text': text_2})

x_train_2, x_test_2, _, _ = train_test_split(
    x_2,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

vectorizer_2 = TfidfVectorizer()

X_train_2 = sp.hstack((
    vectorizer_2.fit_transform(x_train_2['text']),
    x_train_2['date'].to_numpy().reshape(-1, 1),
))

X_test_2 = sp.hstack((
    vectorizer_2.transform(x_test_2['text']),
    x_test_2['date'].to_numpy().reshape(-1, 1),
))

model_2 = RandomForestClassifier(n_estimators=5, n_jobs=10, random_state=RANDOM_STATE)
model_2.fit(X_train_2, y_train)
pred_2 = model_2.predict(X_test_2)

metrics_2 = evaluate_predictions(y_test, pred_2)
metrics_2

{'precision': 0.961, 'recall': 0.96, 'f1': 0.961}

## 3) TF-IDF по `body` с `ngram_range=(2, 2)`

In [6]:
vectorizer_3 = TfidfVectorizer(ngram_range=(2, 2))
X_train_3 = vectorizer_3.fit_transform(x_train_body)
X_test_3 = vectorizer_3.transform(x_test_body)

model_3 = RandomForestClassifier(n_estimators=5, n_jobs=10, random_state=RANDOM_STATE)
model_3.fit(X_train_3, y_train)
pred_3 = model_3.predict(X_test_3)

metrics_3 = evaluate_predictions(y_test, pred_3)
metrics_3

{'precision': 0.951, 'recall': 0.944, 'f1': 0.946}

## Итоговые метрики

In [7]:
results = pd.DataFrame(
    [
        {'experiment': '1) body + TF-IDF', **metrics_1},
        {'experiment': '2) (subject+body) + TF-IDF + date', **metrics_2},
        {'experiment': '3) body + TF-IDF (2,2)', **metrics_3},
    ]
)
results

,experiment,precision,recall,f1
0,1) body + TF-IDF,0.964,0.963,0.963
1,2) (subject+body) + TF-IDF + date,0.961,0.960,0.961
2,"3) body + TF-IDF (2,2)",0.951,0.944,0.946
